## 1. Setup and Data Loading
Imports libraries and loads the protein dataset.

# CNN + Embeddings from ESM-2


In [1]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-pdb-intersect-pisces.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

print(df.head())
df.info()

  pdb_id chain_code                   seq                  sst8  \
0   1FV1          F  NPVVHFFKNIVTPRTPPPSQ  CCCCCBCCCCCCCCCCCCCC   
1   1LM8          H  DLDLEMLAPYIPMDDDFQLR  CCCCCCCCCBCCSCCCEECC   
2   1O06          A  EEDPDLKAAIQESLREAEEA  CCCHHHHHHHHHHHHHHHTC   
3   1QOW          D  CTFTLPGGGGVCTLTSECI*  CCTTSCTTCSSTTSSTTCCC   
4   1RDQ          I  TTYADFIASGRTGRRNAIHD  CHHHHHHTSSCSSCCCCEEC   

                   sst3  len  has_nonstd_aa Exptl.  resolution  R-factor  \
0  CCCCCECCCCCCCCCCCCCC   20          False   XRAY        1.90      0.23   
1  CCCCCCCCCECCCCCCEECC   20          False   XRAY        1.85      0.20   
2  CCCHHHHHHHHHHHHHHHCC   20          False   XRAY        1.45      0.19   
3  CCCCCCCCCCCCCCCCCCCC   20           True   XRAY        1.06      0.14   
4  CHHHHHHCCCCCCCCCCEEC   20          False   XRAY        1.26      0.13   

   FreeRvalue  
0        0.27  
1        0.24  
2        0.22  
3        1.00  
4        0.16  
<class 'pandas.core.frame.DataFrame'>
RangeI

## 2. Load Pre-trained ESM-2 Model
This is used to generate the frozen embeddings.

In [2]:
# Load ESM-2 model and alphabet
esm_model, alphabet = torch.hub.load("facebookresearch/esm:main", "esm2_t30_150M_UR50D")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
esm_model.eval().to(device)
batch_converter = alphabet.get_batch_converter()

# Vocabularies for SST8 and SST3 labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# Get the maximum sequence length from the dataframe
max_len = df["len"].max()

Using cache found in /home/users/ntu/ktang022/.cache/torch/hub/facebookresearch_esm_main


## 3. Generate Frozen Embeddings
Same as your original notebook. This pre-computes all embeddings.

In [3]:
# Prepare data for batching
sequences = [s.replace("*", "X") for s in df['seq'].tolist()]
labels = df['pdb_id'].tolist()
data = list(zip(labels, sequences))

batch_size = 8
all_embeddings = []

for i in tqdm(range(0, len(data), batch_size), desc="Generating Embeddings"):
    batch_data = data[i:i+batch_size]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to(device)
    
    with torch.no_grad():
        results = esm_model(batch_tokens, repr_layers=[esm_model.num_layers], return_contacts=False)
    
    # Extract embeddings and remove start/end tokens
    embeddings = results["representations"][esm_model.num_layers][:, 1:-1, :]
    all_embeddings.extend([emb.cpu() for emb in embeddings])

# Pad embeddings to the maximum length
padded_embeddings = pad_sequence(all_embeddings, batch_first=True, padding_value=0.0)

Generating Embeddings: 100%|██████████| 1135/1135 [01:29<00:00, 12.63it/s]


## 4. Encode Labels and Create Dataset
Same as your original notebook.

In [4]:
def encode_labels(ss_labels, vocab, max_len):
    encoded = []
    for ss in ss_labels:
        ids = [vocab.get(c, -1) for c in ss]
        if len(ids) < max_len:
            ids.extend([-1] * (max_len - len(ids)))
        else:
            ids = ids[:max_len]
        encoded.append(torch.tensor(ids, dtype=torch.long))
    return pad_sequence(encoded, batch_first=True, padding_value=-1)

ss8_labels = encode_labels(df["sst8"], ss8_vocab, max_len)
ss3_labels = encode_labels(df["sst3"], ss3_vocab, max_len)

class ProteinDataset(Dataset):
    def __init__(self, embeddings, sst8_labels, sst3_labels):
        self.embeddings = embeddings
        self.sst8_labels = sst8_labels
        self.sst3_labels = sst3_labels

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.sst8_labels[idx], self.sst3_labels[idx]


## 5. Split Data and Create Dataloaders
Same as your original notebook.

In [5]:
# Split indices for train, validation, and test sets
train_indices, temp_indices = train_test_split(range(len(padded_embeddings)), test_size=0.2, random_state=42)
val_indices, test_indices = train_test_split(temp_indices, test_size=0.5, random_state=42)

# Create datasets
train_dataset = ProteinDataset(padded_embeddings[train_indices], ss8_labels[train_indices], ss3_labels[train_indices])
val_dataset = ProteinDataset(padded_embeddings[val_indices], ss8_labels[val_indices], ss3_labels[val_indices])
test_dataset = ProteinDataset(padded_embeddings[test_indices], ss8_labels[test_indices], ss3_labels[test_indices])

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

embedding_dim = padded_embeddings.shape[-1]

## 6. Define the CNN Model
**MODIFIED:** This cell defines a 1D CNN model instead of a Transformer. It's designed to accept the exact same input shape `(batch_size, seq_len, embedding_dim)`.

In [6]:
class ProteinCNN(nn.Module):
    def __init__(self, input_dim=640, num_filters=128, dropout=0.1):
        super().__init__()
        
        # 1D Convolutional layers
        # nn.Conv1d expects input as (batch, channels, length)
        # Our input is (batch, length, channels), so we'll permute it.
        self.conv1 = nn.Conv1d(in_channels=input_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        
        self.conv3 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters, kernel_size=7, padding=3)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)

        # Two separate classifier heads
        self.q8_head = nn.Linear(num_filters, 8)
        self.q3_head = nn.Linear(num_filters, 3)

    def forward(self, x, mask=None): # Mask is not used by CNN, but kept for compatibility
        """
        x: [batch_size, seq_len, input_dim] (embeddings)
        """
        # Permute from [batch, seq_len, channels] to [batch, channels, seq_len]
        x = x.permute(0, 2, 1)
        
        x = self.dropout1(self.relu1(self.conv1(x)))
        x = self.dropout2(self.relu2(self.conv2(x)))
        x = self.dropout3(self.relu3(self.conv3(x)))
        
        # Permute back to [batch, seq_len, channels]
        x = x.permute(0, 2, 1)
        
        # Per-residue classification
        q8_logits = self.q8_head(x)
        q3_logits = self.q3_head(x)
        
        return q8_logits, q3_logits

## 7. Training Loop
**MODIFIED:** This now instantiates `ProteinCNN` instead of `ProteinTransformer`. The rest of the logic is identical, as the inputs and outputs are the same.

In [7]:
def compute_accuracy(pred_logits, labels):
    """Per-residue accuracy ignoring -1 padding"""
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# MODIFIED: Initialize the CNN model
model = ProteinCNN(input_dim=embedding_dim)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
model.to(device)

# Losses and optimizer (same as before)
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 20
best_val_acc_q8 = 0.0

for epoch in range(num_epochs):
    model.train()
    train_loss, train_acc_q8, train_acc_q3 = 0, 0, 0
    
    for embeddings, ss8, ss3 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
        
        # NOTE: The mask is not used by the CNN, but we pass it for compatibility.
        # The original padding mask (ss8 == -1) is still used for loss calculation.
        mask = (ss8 == -1)
        
        # Forward pass
        q8_logits, q3_logits = model(embeddings, mask=None)
        
        # Loss
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_acc_q8 += compute_accuracy(q8_logits, ss8)
        train_acc_q3 += compute_accuracy(q3_logits, ss3)
    
    train_loss /= len(train_loader)
    train_acc_q8 /= len(train_loader)
    train_acc_q3 /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss, val_acc_q8, val_acc_q3 = 0, 0, 0
    with torch.no_grad():
        for embeddings, ss8, ss3 in val_loader:
            embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
            mask = (ss8 == -1)
            q8_logits, q3_logits = model(embeddings, mask=None)
            
            loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
            loss = loss_q8 + 0.5 * loss_q3
            
            val_loss += loss.item()
            val_acc_q8 += compute_accuracy(q8_logits, ss8)
            val_acc_q3 += compute_accuracy(q3_logits, ss3)
    
    val_loss /= len(val_loader)
    val_acc_q8 /= len(val_loader)
    val_acc_q3 /= len(val_loader)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    print(f"Train Acc Q8={train_acc_q8:.4f}, Val Acc Q8={val_acc_q8:.4f}")
    print(f"Train Acc Q3={train_acc_q3:.4f}, Val Acc Q3={val_acc_q3:.4f}")
    
    # Save best model
    if val_acc_q8 > best_val_acc_q8:
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save(model_state, "best_cnn_model.pt")
        best_val_acc_q8 = val_acc_q8

Epoch 1/20: 100%|██████████| 454/454 [00:09<00:00, 45.41it/s]


Epoch 1: Train Loss=1.3057, Val Loss=1.0112
Train Acc Q8=0.6330, Val Acc Q8=0.7134
Train Acc Q3=0.7643, Val Acc Q3=0.8367


Epoch 2/20: 100%|██████████| 454/454 [00:07<00:00, 58.52it/s]


Epoch 2: Train Loss=0.9981, Val Loss=0.9630
Train Acc Q8=0.7177, Val Acc Q8=0.7253
Train Acc Q3=0.8362, Val Acc Q3=0.8430


Epoch 3/20: 100%|██████████| 454/454 [00:07<00:00, 57.83it/s]


...

Epoch 18: Train Loss=0.8294, Val Loss=0.8775
Train Acc Q8=0.7627, Val Acc Q8=0.7510
Train Acc Q3=0.8638, Val Acc Q3=0.8555


Epoch 19/20: 100%|██████████| 454/454 [00:20<00:00, 21.66it/s]


Epoch 19: Train Loss=0.8266, Val Loss=0.8769
Train Acc Q8=0.7634, Val Acc Q8=0.7513
Train Acc Q3=0.8639, Val Acc Q3=0.8564


Epoch 20/20: 100%|██████████| 454/454 [00:20<00:00, 21.69it/s]


Epoch 20: Train Loss=0.8212, Val Loss=0.8813
Train Acc Q8=0.7649, Val Acc Q8=0.7507
Train Acc Q3=0.8652, Val Acc Q3=0.8556


## 8. Final Evaluation on Test Set
**MODIFIED:** Loads the saved `best_cnn_model.pt`.

In [8]:
# Initialize a new model instance
model = ProteinCNN(input_dim=embedding_dim)
# Load the best model state
model.load_state_dict(torch.load("best_cnn_model.pt"))

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
model.eval()

test_loss, test_acc_q8, test_acc_q3 = 0, 0, 0
with torch.no_grad():
    for embeddings, ss8, ss3 in test_loader:
        embeddings, ss8, ss3 = embeddings.to(device), ss8.to(device), ss3.to(device)
        q8_logits, q3_logits = model(embeddings, mask=None)
        
        loss_q8 = criterion_q8(q8_logits.view(-1, 8), ss8.view(-1))
        loss_q3 = criterion_q3(q3_logits.view(-1, 3), ss3.view(-1))
        loss = loss_q8 + 0.5 * loss_q3
        
        test_loss += loss.item()
        test_acc_q8 += compute_accuracy(q8_logits, ss8)
        test_acc_q3 += compute_accuracy(q3_logits, ss3)

test_loss /= len(test_loader)
test_acc_q8 /= len(test_loader)
test_acc_q3 /= len(test_loader)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy Q8: {test_acc_q8:.4f}")
print(f"Test Accuracy Q3: {test_acc_q3:.4f}")

Test Loss: 0.8944
Test Accuracy Q8: 0.7471
Test Accuracy Q3: 0.8524
